# 05 · Retrieval evaluation
Step 6: score retrieval on the 39 answerable gold questions for every chunking strategy (fixed / semantic) and retrieval mode (dense / BM25 / hybrid), through the agent's real analyze → decompose → retrieve steps.

**Cost:** 39 query-analysis calls (`gpt-4.1`, ~1K tokens each, cached in `eval/cache/` after the first run) plus query embeddings (cached). A few cents the first time, about zero after.

**Metrics:** hit@k (any evidence in top k), recall@k (share of evidence items found, matters for comparisons), all@k (every evidence item found), MRR (1 / rank of the first relevant chunk). k = 3, 6, 12; the agent passes 12 chunks to the answer step.

Shut down other notebook kernels first if they have the index open.

In [1]:
import sys, pathlib
print(sys.executable)
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
pd.set_option('display.width', 200)
from finsight.evaluation.retrieval import run_retrieval_eval, summarize, load_gold, analyze_all

c:\Users\senpr\Documents\finsight-ai\.venv\Scripts\python.exe


## 1. Query analysis check
Did the agent extract the right companies and years? Compare against the evidence documents.

In [2]:
gold = [g for g in load_gold() if g.get('evidence')]
analyses = analyze_all(gold)
rows = []
for g in gold:
    a = analyses[g['id']]
    ev_docs = sorted({d for e in g['evidence'] for d in e['docs']})
    rows.append({'id': g['id'], 'question': g['question'][:60], 'companies': a['companies'], 'years': a['fiscal_years'],
                 'in_scope': a['in_scope'], 'evidence_docs': ev_docs})
pd.DataFrame(rows)

,id,question,companies,years,in_scope,evidence_docs
0,L01,What was AWS net sales in 2024?,[Amazon],[2024],True,[amazon_2024]
1,L02,What was AWS operating income in 2024?,[Amazon],[2024],True,[amazon_2024]
2,L03,What were Amazon's total net sales in 2025?,[Amazon],[2025],True,[amazon_2025]
3,L04,What was Amazon's net income in 2023?,[Amazon],[2023],True,[amazon_2023]
4,L05,What was Amazon's free cash flow in 2025?,[Amazon],[2025],True,[amazon_2025]
5,L06,What were Amazon's diluted earnings per share ...,[Amazon],[2024],True,[amazon_2024]
6,L07,How much did Amazon spend on purchases of prop...,[Amazon],[2025],True,[amazon_2025]
7,L08,What were Amazon's North America segment net s...,[Amazon],[2023],True,[amazon_2023]
8,L09,How much revenue did Google Cloud generate in ...,[Alphabet],[2025],True,[alphabet_2025]
9,L10,What was Google Cloud operating income in 2025?,[Alphabet],[2025],True,[alphabet_2025]


## 2. Run the grid

In [3]:
df = run_retrieval_eval()
print(len(df), 'rows saved to eval/reports/')
summarize(df)

336 rows saved to eval/reports/


hit@3  hit@6  hit@12  recall@3  recall@6  recall@12  all@6    mrr
strategy mode                                                                     
fixed    bm25    0.554  0.732   0.839     0.548     0.723      0.812  0.714  0.443
         dense   0.411  0.625   0.768     0.390     0.601      0.741  0.589  0.357
         hybrid  0.518  0.625   0.839     0.485     0.604      0.810  0.571  0.419
semantic bm25    0.571  0.714   0.839     0.545     0.705      0.812  0.696  0.465
         dense   0.446  0.643   0.714     0.405     0.610      0.690  0.589  0.331
         hybrid  0.482  0.661   0.857     0.440     0.634      0.836  0.607  0.381

## 3. By question type (hybrid)

In [4]:
summarize(df[df['mode'] == 'hybrid'], by=('strategy', 'type'))

hit@3  hit@6  hit@12  recall@3  recall@6  recall@12  all@6    mrr
strategy type                                                                         
fixed    comparison  0.500  0.600   0.900     0.317     0.483      0.733  0.300  0.451
         followup    0.500  1.000   1.000     0.500     1.000      1.000  1.000  0.350
         lookup      0.459  0.541   0.784     0.459     0.541      0.784  0.541  0.394
         narrative   0.857  1.000   1.000     0.857     1.000      1.000  1.000  0.524
semantic comparison  0.600  0.600   0.800     0.367     0.450      0.683  0.300  0.467
         followup    1.000  1.000   1.000     1.000     1.000      1.000  1.000  0.417
         lookup      0.432  0.622   0.838     0.432     0.622      0.838  0.622  0.371
         narrative   0.429  0.857   1.000     0.429     0.857      1.000  0.857  0.299

## 4. Misses
Questions where hybrid retrieval did not find all evidence in the top 12.

In [5]:
miss = df[(df['mode'] == 'hybrid') & (df['all@12'] < 1)][['id', 'type', 'strategy', 'companies', 'years', 'recall@12', 'first_rank', 'top_chunks']]
miss

,id,type,strategy,companies,years,recall@12,first_rank,top_chunks
135,L24,lookup,fixed,Microsoft,,0.000000,NaN,microsoft_2024-fixed-00050 microsoft_2025-fixe...
136,C01,comparison,fixed,"Amazon,Alphabet",2025,0.500000,4.0,amazon_2025-fixed-00007 alphabet_2025-fixed-00...
140,C05,comparison,fixed,"Microsoft,Amazon,Alphabet",2024,0.666667,2.0,microsoft_2024-fixed-00055 amazon_2024-table-0...
143,C08,comparison,fixed,Microsoft,"2023,2024",0.000000,NaN,microsoft_2023-fixed-00056 microsoft_2024-fixe...
151,P01,lookup,fixed,Amazon,2025,0.000000,NaN,amazon_2025-fixed-00085 amazon_2025-fixed-0007...
153,P03,lookup,fixed,Amazon,2024,0.000000,NaN,amazon_2024-fixed-00168 amazon_2024-fixed-0002...
159,P09,lookup,fixed,Alphabet,2024,0.000000,NaN,alphabet_2024-fixed-00167 alphabet_2024-fixed-...
160,P10,lookup,fixed,Microsoft,2024,0.000000,NaN,microsoft_2024-fixed-00032 microsoft_2024-fixe...
161,P11,lookup,fixed,Microsoft,2025,0.000000,NaN,microsoft_2025-fixed-00041 microsoft_2025-fixe...
162,P12,lookup,fixed,Microsoft,2025,0.000000,NaN,microsoft_2025-fixed-00041 microsoft_2025-fixe...


In [6]:
g_by_id = {g['id']: g for g in gold}
for i in miss['id'].unique():
    print(i, '|', g_by_id[i]['question'], '| evidence:', g_by_id[i]['evidence'])

L24 | What was Microsoft's total revenue in fiscal year 2022? | evidence: [{'docs': ['microsoft_2023', 'microsoft_2024'], 'contains': ['198,270']}]
C01 | Compare AWS and Google Cloud revenue in 2025. | evidence: [{'docs': ['amazon_2025'], 'contains': ['AWS', '128,725']}, {'docs': ['alphabet_2025'], 'contains': ['Google Cloud', '58,705']}]
C05 | Compare the net income of Microsoft, Amazon and Alphabet in 2024. | evidence: [{'docs': ['microsoft_2024'], 'contains': ['Net income', '88,136']}, {'docs': ['amazon_2024'], 'contains': ['Net income', '59,248']}, {'docs': ['alphabet_2024'], 'contains': ['Net income', '100,118']}]
C08 | Compare Microsoft's Intelligent Cloud revenue in fiscal years 2023 and 2024. | evidence: [{'docs': ['microsoft_2023', 'microsoft_2024'], 'contains': ['Intelligent Cloud', '87,907']}, {'docs': ['microsoft_2024'], 'contains': ['Intelligent Cloud', '105,362']}]
P01 | What was Amazon's capex in 2025? | evidence: [{'docs': ['amazon_2025'], 'contains': ['Purchases of pro

## 5. Hybrid weight sweep
Dense vs BM25 balance in the fusion. Analyses and query embeddings are cached, so this costs nothing. Compare with the BM25-only rows.

In [7]:
sweep = []
for w in [(0.5, 0.5), (0.3, 0.7), (0.2, 0.8)]:
    d = run_retrieval_eval(modes=('hybrid',), weights=w, tag=f'hybrid_w{w[0]}')
    s = summarize(d).reset_index(); s.insert(2, 'weights (dense,bm25)', str(w)); sweep.append(s)
bm = summarize(df[df['mode'] == 'bm25']).reset_index(); bm.insert(2, 'weights (dense,bm25)', 'bm25 only')
pd.concat(sweep + [bm]).set_index(['strategy', 'weights (dense,bm25)']).sort_index()

mode  hit@3  hit@6  hit@12  recall@3  recall@6  recall@12  all@6    mrr
strategy weights (dense,bm25)                                                                           
fixed    (0.2, 0.8)            hybrid  0.482  0.696   0.839     0.464     0.676      0.801  0.643  0.441
         (0.3, 0.7)            hybrid  0.500  0.643   0.821     0.482     0.622      0.774  0.589  0.448
         (0.5, 0.5)            hybrid  0.518  0.625   0.839     0.485     0.604      0.810  0.571  0.419
         bm25 only               bm25  0.554  0.732   0.839     0.548     0.723      0.812  0.714  0.443
semantic (0.2, 0.8)            hybrid  0.500  0.732   0.857     0.464     0.711      0.818  0.679  0.428
         (0.3, 0.7)            hybrid  0.446  0.714   0.839     0.411     0.693      0.810  0.661  0.404
         (0.5, 0.5)            hybrid  0.482  0.661   0.857     0.440     0.634      0.836  0.607  0.381
         bm25 only               bm25  0.571  0.714   0.839     0.545     0.705      0.812  0.696  0.465

## Notes / findings
_(fill in after reviewing)_